# Final Model Selection & Artifact Export

## 1. Goal
Train final models on all data and export inference artifacts to `backend/app/ai/artifacts/`.

In [1]:
import os
import sys
import json
import joblib
from pathlib import Path
import pandas as pd
from datetime import datetime

project_root = Path(os.getcwd()).parent.parent
sys.path.append(str(project_root))

from notebooks.config import *


## 2. Load Final Data & Features

In [2]:
df = pd.read_parquet(CLEANED_DATA_PATH)
features = ['latitude', 'longitude', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_peak_hour', 'priority_encoded', 'is_closed']
X = df[features]


## 3. Train Final Models

In [3]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import xgboost as xgb

# Example: Congestion Proxy (Regressor)
congestion_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
congestion_model.fit(X, df[CONGESTION_TARGET])

# Example: Response Time (Regressor) - Filter out NaN resolution times
response_mask = df[RESPONSE_TIME_TARGET].notna()
X_resp = X[response_mask]
y_resp = df.loc[response_mask, RESPONSE_TIME_TARGET]

response_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
response_model.fit(X_resp, y_resp)

# Example: Deployment (Classifier)
deployment_model = RandomForestClassifier(n_estimators=100, random_state=42)
deployment_model.fit(X, df[DEPLOYMENT_TARGET])

print("Final models trained.")


Final models trained.


## 4. Export Artifacts

In [4]:
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Save Models
joblib.dump(congestion_model, ARTIFACTS_DIR / "congestion_model.pkl")
joblib.dump(response_model, ARTIFACTS_DIR / "response_time_model.pkl")
joblib.dump(deployment_model, ARTIFACTS_DIR / "deployment_model.pkl")

# Save Registry
registry = {
    "version": "1.0",
    "trained_at": datetime.utcnow().isoformat(),
    "features": features,
    "metrics": {
        "congestion_rmse": 0.0, # Placeholder for real metrics
        "response_rmse": 0.0,
        "deployment_accuracy": 0.0
    }
}

with open(ARTIFACTS_DIR / "model_registry.json", "w") as f:
    json.dump(registry, f, indent=4)

print("Exported models and registry to backend artifacts.")


Exported models and registry to backend artifacts.
